In [1]:
%store -r

In [2]:
import boto3
import sagemaker
import pickle
import zarr
from zarr.codecs import ZstdCodec
import pandas
from pathlib import Path
import pandas as pd
import numpy as np
from tqdm import tqdm
import s3fs
import os
import shutil
import kagglehub

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [3]:
session = boto3.Session()
region = session.region_name
s3 = session.client(service_name="s3", region_name=region)
default_bucket = default_bucket
s3_project_root = s3_project_root

# Initial Dataset Download

In [4]:
# Download latest version
path = kagglehub.dataset_download("mohamedalqblawi/rplan-pickle-files")
print("Path to dataset files:", path)

Path to dataset files: /home/sagemaker-user/.cache/kagglehub/datasets/mohamedalqblawi/rplan-pickle-files/versions/1


# Deserializing and Uploading to S3

In [5]:
# Deserializing settings
BATCH_SIZE = 1280
COMPRESSORS = ZstdCodec()
LOCAL_DATA_DIR = Path("../.cache/kagglehub/datasets/mohamedalqblawi/rplan-pickle-files/versions/1/pickle/")
ZARR_REF = "data.zarr"
METADATA_REF = "metadata.parquet"

In [6]:
pickles = list(LOCAL_DATA_DIR.rglob("*.pkl"))
store = zarr.open(ZARR_REF, mode="w")


# batch data
inside_masks, boundary_masks, room_masks, door_masks, meta_batch, id_batch = [], [], [], [], [], []

# zarr datasets, initialized on first write
inside_masks_ds, boundary_masks_ds, room_masks_ds, door_masks_ds, ids_ds = None, None, None, None, None

sample_counter = 0
room_metadata = []

def flush_batch(im_batch, bm_batch, rm_batch, dr_batch, id_batch):
    global inside_masks_ds, boundary_masks_ds, room_masks_ds, door_masks_ds, ids_ds

    im_arr = np.stack(im_batch)
    bm_arr = np.stack(bm_batch)
    rm_arr = np.stack(rm_batch)
    dr_arr = np.stack(dr_batch)
    ids_arr = np.array(id_batch, dtype=str)

    if inside_masks_ds is None:
        # First flush — create arrays with explicit shape/dtype, data as keyword arg
        inside_masks_ds = store.create_array(
            "inside_masks",
            chunks=(BATCH_SIZE, *im_arr.shape[1:]),
            compressors=COMPRESSORS,
            data=im_arr
        )
        boundary_masks_ds = store.create_array(
            "boundary_masks",
            chunks=(BATCH_SIZE, *bm_arr.shape[1:]),
            compressors=COMPRESSORS,
            data=bm_arr
        )
        room_masks_ds = store.create_array(
            "room_masks",
            chunks=(BATCH_SIZE, *rm_arr.shape[1:]),
            compressors=COMPRESSORS,
            data=rm_arr
        )
        door_masks_ds = store.create_array(
            "door_masks",
            chunks=(BATCH_SIZE, *dr_arr.shape[1:]),
            compressors=COMPRESSORS,
            data=dr_arr
        )
        ids_ds = store.create_array(
            "sample_ids",
            dtype='str',
            shape=ids_arr.shape
        )
        ids_ds.append(ids_arr)
    else:
        # Subsequent flushes — append along axis 0
        inside_masks_ds.append(im_arr)
        boundary_masks_ds.append(bm_arr)
        room_masks_ds.append(rm_arr)
        door_masks_ds.append(dr_arr)
        ids_ds.append(ids_arr)

for pkl_path in tqdm(pickles, "Deserializing pickles...", len(pickles), unit="pickles"):
    with open(pkl_path, "rb") as f:
        i_mask, b_mask, r_mask, d_mask, meta = pickle.load(f)
        
    inside_masks.append(i_mask)
    boundary_masks.append(b_mask//127) # original data encodes walls as 127, doors as 255
    room_masks.append(r_mask)
    door_masks.append(d_mask)
    id_batch.append(str(pkl_path).split("/")[-1])
    meta_batch.append(meta)
    
    sample_counter += 1

    if len(inside_masks) >= BATCH_SIZE:
        flush_batch(inside_masks, boundary_masks, room_masks, door_masks, id_batch)
        room_metadata.extend(meta_batch)
        inside_masks, boundary_masks, room_masks, door_masks, id_batch, meta_batch = [], [], [], [], [], []

# Flush any remaining samples that didn't fill a full batch
if inside_masks:
    flush_batch(inside_masks, boundary_masks, room_masks, door_masks, id_batch)
    room_metadata.extend(meta_batch)

# writing metadata
df = pd.DataFrame(room_metadata)
df["id"] = [str(pkl).split("/")[-1] for pkl in pickles]
df.insert(0, "sample_id", [f"sample_{i:05d}" for i in range(sample_counter)])
df.to_parquet(METADATA_REF, index=False)

print(f"Done. {sample_counter} samples written.")

Deserializing pickles...:   0%|          | 0/80788 [00:00<?, ?pickles/s]/tmp/ipykernel_3948/2095241029.py:65: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  i_mask, b_mask, r_mask, d_mask, meta = pickle.load(f)
Deserializing pickles...: 100%|██████████| 80788/80788 [09:09<00:00, 147.02pickles/s]


Done. 80788 samples written.


In [7]:
datastore_dest_path = f"{s3_project_root}/store"
metadata_dest_path = f"{s3_project_root}/metadata"

# Uses your AWS credentials automatically (~/.aws/credentials or env vars)
fs = s3fs.S3FileSystem()

# Upload metadata to S3
fs.put(METADATA_REF, f"{default_bucket}/{metadata_dest_path}/{METADATA_REF}")

# Upload data store
fs.put(ZARR_REF, f"{default_bucket}/{datastore_dest_path}/{ZARR_REF}", recursive=True)

[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,

# Clean up local

In [8]:
if Path(METADATA_REF).exists():
    os.remove(METADATA_REF)

if Path(ZARR_REF).exists():
    shutil.rmtree(ZARR_REF)